# Train Rwanda Travel Fare Model

This notebook trains the local fare prediction model from the `Routes` sheet inside `data/routes.xlsx`.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = Path("../data/routes.xlsx")
MODEL_PATH = Path("../backend/ml_model.pkl")

df = pd.read_excel(DATA_PATH, sheet_name="Routes", engine="openpyxl")
df.head()

In [ ]:
feature_columns = ["from_city", "to_city", "Distance_km", "transport_type", "demand"]
target_column = "price_rwf"

X = df[feature_columns]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            ["from_city", "to_city", "transport_type", "demand"],
        ),
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=250, random_state=42)),
    ]
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, predictions), 2))
print("R2:", round(r2_score(y_test, predictions), 4))

In [ ]:
joblib.dump(model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH.resolve()}")